In [1]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

In [2]:
load_dotenv()

api_key = os.getenv("CLAUDE_API_KEY")
if not api_key:
    raise ValueError("CLAUDE_API_KEY não encontrada no arquivo .env")

client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-5"

In [3]:
df = pd.read_csv("../data/raw/dataset.csv")
df.head()

,ID,Text,FT,FP,FR,FS,GP1,GP2,GP3,GP4,GP5,GP6,GP7,Type,Course,Intructor
0,20,Ter cuidado com as cópias fieis da internet. P...,0,1,0,0,0,0,0,0,0,0,0,Individual,literature,1
1,22,"Texto bem escrito, apenas rever as margens (co...",0,1,0,0,0,0,0,0,0,1,0,Individual,literature,1
2,23,Ler outros textos sobre o assunto para lher da...,1,0,0,0,0,0,0,0,0,1,0,Individual,literature,1
3,24,"Procure ler mais, para escrever mais acerca do...",1,1,0,0,0,0,0,0,0,1,0,Individual,literature,1
4,25,"Não realizar cópia da internet na íntegra, pro...",0,1,0,0,0,0,0,0,0,0,0,Individual,literature,1


In [4]:
texto = df["Text"].iloc[0]
print(texto)

Ter cuidado com as cópias fieis da internet. Procurar ler e escrever com suas palavras a partir do que entendeu.


In [5]:
def montar_prompt(texto: str) -> str:
    return f"""
Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
{texto}
"""

In [6]:
prompt = montar_prompt(texto)
print(prompt)


Você é um especialista em avaliação educacional.

Classifique o feedback nas categorias abaixo. Para cada uma, responda 1 (presente) ou 0 (ausente).

GP1: Esclarece o que é um bom desempenho.
GP2: Promove autoavaliação e reflexão do aluno.
GP3: Fornece informações de qualidade sobre o aprendizado.
GP4: Incentiva diálogo entre aluno e professor/colegas.
GP5: Incentiva motivação e autoestima.
GP6: Ajuda a reduzir a diferença entre desempenho atual e desejado.
GP7: Fornece informações úteis para o professor ajustar o ensino.

Responda no formato:
GP1: 0 ou 1
GP2: 0 ou 1
GP3: 0 ou 1
GP4: 0 ou 1
GP5: 0 ou 1
GP6: 0 ou 1
GP7: 0 ou 1

Feedback:
Ter cuidado com as cópias fieis da internet. Procurar ler e escrever com suas palavras a partir do que entendeu.



In [7]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}],
)
resposta = next(bloco.text for bloco in response.content if bloco.type == "text")
print(resposta)

GP1: 0
GP2: 0
GP3: 1
GP4: 0
GP5: 0
GP6: 1
GP7: 0


## Processamento em lote (Batch API)

A Anthropic oferece a [Message Batches API](https://platform.claude.com/docs/en/build-with-claude/batch-processing), que permite enviar várias mensagens de uma vez em vez de uma requisição por linha do dataset.

Vantagens:
- **50% mais barato** que chamadas individuais (preço por token reduzido pela metade).
- Processa até 100.000 requisições por lote, de forma assíncrona.
- Ideal para o nosso caso: o dataset tem 1000 feedbacks — chamar a API um por um seria lento e mais caro.

O fluxo é: (1) montar uma lista de requisições, cada uma com um `custom_id` único; (2) criar o lote; (3) aguardar o processamento; (4) buscar os resultados e casar cada resposta com seu `custom_id` (a ordem das respostas não é garantida, por isso o `custom_id` é essencial).

**Abaixo, o lote roda com o dataset completo (1000 feedbacks).** O tempo de processamento não é garantido — pode levar de poucos minutos até (no pior caso) 24h. Se quiser testar mais rápido antes de rodar tudo, troque `linhas = df` por `linhas = df.head(5)`.

In [ ]:
linhas = df  # dataset completo (1000 feedbacks). Para testar rápido, use: df.head(5)

requests = [
    Request(
        custom_id=f"feedback-{row.ID}",
        params=MessageCreateParamsNonStreaming(
            model=MODEL,
            max_tokens=1024,
            messages=[{"role": "user", "content": montar_prompt(row.Text)}],
        ),
    )
    for row in linhas.itertuples()
]

batch = client.messages.batches.create(requests=requests)
print(f"Lote criado: {batch.id} (status: {batch.processing_status}, {len(requests)} requisições)")

In [ ]:
# Aguarda o lote terminar de processar (consulta o status a cada 15s)
while True:
    batch = client.messages.batches.retrieve(batch.id)
    if batch.processing_status == "ended":
        break
    print(f"Processando... {batch.request_counts}")
    time.sleep(15)

print("Lote concluído!")
print(batch.request_counts)

In [ ]:
resultados = {}
for result in client.messages.batches.results(batch.id):
    if result.result.type == "succeeded":
        msg = result.result.message
        resultados[result.custom_id] = next(
            bloco.text for bloco in msg.content if bloco.type == "text"
        )
    else:
        resultados[result.custom_id] = f"[erro: {result.result.type}]"

for custom_id in sorted(resultados):
    print(f"--- {custom_id} ---")
    print(resultados[custom_id])
    print()